In [ ]:
from pyspark.sql.functions import col, current_timestamp, expr, when

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_orders_table_name = dbutils.widgets.get("raw_olist_orders_table")

silver_schema = dbutils.widgets.get("silver_schema")
orders_table_name = dbutils.widgets.get("orders_table")

In [ ]:
raw_olist_orders_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_orders_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{orders_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{orders_table_name} (
            orderId STRING,
            customerId STRING,
            orderStatus STRING,
            orderPurchaseTimestamp TIMESTAMP,
            orderApprovedAt TIMESTAMP,
            orderDeliveredCarrierDate TIMESTAMP,
            orderDeliveredCustomerDate TIMESTAMP,
            orderEstimatedDeliveryDate TIMESTAMP,
            processedTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
orders_silver_df = (
    raw_olist_orders_df.where((col("order_id").rlike("^[0-9a-fA-F]{32}$")))
    .select(
        col("order_id").cast("string").alias("orderId"),
        when(col("customer_id").rlike("^[0-9a-fA-F]{32}$"), col("customer_id"))
        .otherwise(None)
        .cast("string")
        .alias("customerId"),
        col("order_status").cast("string").alias("orderStatus"),
        expr("try_to_timestamp(order_purchase_timestamp, 'yyyy-MM-dd HH:mm:ss')").alias("orderPurchaseTimestamp"),
        expr("try_to_timestamp(order_approved_at, 'yyyy-MM-dd HH:mm:ss')").alias("orderApprovedAt"),
        expr("try_to_timestamp(order_delivered_carrier_date, 'yyyy-MM-dd HH:mm:ss')").alias(
            "orderDeliveredCarrierDate"
        ),
        expr("try_to_timestamp(order_delivered_customer_date, 'yyyy-MM-dd HH:mm:ss')").alias(
            "orderDeliveredCustomerDate"
        ),
        expr("try_to_timestamp(order_estimated_delivery_date, 'yyyy-MM-dd HH:mm:ss')").alias(
            "orderEstimatedDeliveryDate"
        ),
    )
    .withColumn("processedTimestamp", current_timestamp())
    .dropDuplicates(["orderId"])
)

In [ ]:
orders_silver_df.createOrReplaceTempView("orders_silver_view")

spark.sql(f"""
    MERGE INTO {catalog}.{silver_schema}.{orders_table_name} AS target
    USING orders_silver_view AS source
    ON target.orderId = source.orderId
    WHEN MATCHED THEN
        UPDATE SET
            target.customerId = source.customerId,
            target.orderStatus = source.orderStatus,
            target.orderPurchaseTimestamp = source.orderPurchaseTimestamp,
            target.orderApprovedAt = source.orderApprovedAt,
            target.orderDeliveredCarrierDate = source.orderDeliveredCarrierDate,
            target.orderDeliveredCustomerDate = source.orderDeliveredCustomerDate,
            target.orderEstimatedDeliveryDate = source.orderEstimatedDeliveryDate,
            target.processedTimestamp = source.processedTimestamp
    WHEN NOT MATCHED THEN
        INSERT *
    """)